
# DENSE LeJEPA PRE-TRAINING — YOLOv8 Multi-Scale Backbone




In [4]:

import os
import re
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import tqdm
from torchvision.transforms import v2
from torchvision.transforms.v2 import functional as TF

from sklearn.decomposition import PCA
import plotly.express as px
from ultralytics import YOLO

# DEVICE SETUP (Apple Silicon MPS Acceleration)

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print(f" Device: Apple Silicon GPU via MPS ({DEVICE})")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f" Device: CUDA ({DEVICE})")
else:
    DEVICE = torch.device("cpu")
    print(f"Device: CPU ({DEVICE})")



# CONFIGURATION & HYPERPARAMETERS

JPEG_ROOT = r"/Users/akintanoreofeoluwa/Downloads/LeJEPA _pretrainining_multi_camera_boll/mars_multi_camera_boll"
CHECKPOINT_DIR = r"./checkpoints_cam235_50k_dense_lejepa"

# 0-Indexed Filename Mapping for 2nd, 3rd, and 5th Physical Cameras:
# 2nd Camera -> _cam1_ | 3rd Camera -> _cam2_ | 5th Camera -> _cam4_
TARGET_CAMERAS_0INDEX = {1, 2, 4}

TARGET_DATASET_SIZE = 50000
IMAGE_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 60
V = 2                          # Number of augmented views generated per sample

PROJ_DIM = 128                 # Per-pixel projection channel width
LR = 1e-3
WEIGHT_DECAY = 1e-4

# Regularization balancing parameter.
# Dense, per-location embeddings are more collapse-prone than a single
# pooled vector, so we lean slightly higher than a typical global-LeJEPA
# lambda. 0.2 is the default; 0.15 is a reasonable fallback if SIGReg
# starts dominating and prediction loss stalls.
LAMBDA = 0.2

# Caps the number of spatial locations fed into SIGReg per scale, per
# image, per view. Keeps the O(N) statistic computation bounded even if
# you later raise IMAGE_SIZE (which grows P3/P4/P5 resolution).
MAX_SIGREG_LOCATIONS = 64

# macOS DataLoader stability setting
NUM_WORKERS = 0
PIN_MEMORY = False

# Log 3D PCA embeddings at these epochs
PCA_EPOCHS = {0, 29, EPOCHS - 1}



# SIGREG REGULARIZATION MODULE (unchanged math, generic input shape)

class SIGReg(nn.Module):
    def __init__(self, knots=17):
        super().__init__()
        t = torch.linspace(0, 3, knots)
        dt = 3 / (knots - 1)
        weights = torch.full((knots,), 2 * dt)
        weights[[0, -1]] = dt
        window = torch.exp(-t.square() / 2.0)

        self.register_buffer("t", t)
        self.register_buffer("phi", window)
        self.register_buffer("weights", weights * window)

    def forward(self, proj):
        # Expects (V, N, D): V views, N i.i.d. samples (here: pixel
        # locations flattened across batch and space), D channels.
        if proj.dim() == 2:
            proj = proj.unsqueeze(0)

        Vv, N, D = proj.shape
        A = torch.randn(D, 256, device=proj.device)
        A = A / (A.norm(dim=0, keepdim=True) + 1e-12)

        x_t = (proj @ A).unsqueeze(-1) * self.t
        err = (x_t.cos().mean(-3) - self.phi).square() + x_t.sin().mean(-3).square()
        statistic = (err @ self.weights) * N
        return statistic.mean()



# NON-GEOMETRIC AUGMENTATIONS
# (Grayscale OR Blur, chosen at random, plus light additive noise.
#  No rotation/flip — spatial grid must stay aligned across views
#  for the dense per-pixel centroid loss to be valid.)
class RandomGrayscaleOrBlur:
    def __init__(self, grayscale_p=0.5, blur_kernel=5, blur_sigma=(0.3, 1.5)):
        self.grayscale_p = grayscale_p
        self.blur_kernel = blur_kernel
        self.blur_sigma = blur_sigma

    def __call__(self, img: torch.Tensor) -> torch.Tensor:
        if torch.rand(1).item() < self.grayscale_p:
            return TF.rgb_to_grayscale(img, num_output_channels=3)
        return TF.gaussian_blur(img, kernel_size=self.blur_kernel, sigma=self.blur_sigma)


class AddLightGaussianNoise:
    """Very light additive noise — perturbs pixel values slightly without
    destroying local structure. Operates on float tensors in [0, 1]."""
    def __init__(self, std=0.02):
        self.std = std

    def __call__(self, img: torch.Tensor) -> torch.Tensor:
        noise = torch.randn_like(img) * self.std
        return (img + noise).clamp(0.0, 1.0)


# DATASET MODULE (0-INDEXED CAMERA 50k SAMPLER)

class Cotton50kSamplerDataset(Dataset):
    """
    Parses MARS robot dataset filename format: clip<num>_cam<num>_frame<num>.jpg
    Handles 0-indexed camera tags (0-5) and samples ~50,000 images equally across 3 camera views.
    """
    def __init__(self, root, target_cameras={1, 2, 4}, target_size=50000, seed=42):
        root = Path(root)
        if not root.exists():
            raise FileNotFoundError(f"JPEG_ROOT path not found: {root}")

        valid_exts = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
        all_image_paths = [Path(p) for p in root.rglob("*") if p.suffix.lower() in valid_exts]

        camera_buckets = {cam_id: [] for cam_id in target_cameras}

        for path in all_image_paths:
            match = re.search(r"_cam([0-5])_", path.name.lower())
            if not match:
                match = re.search(r"cam(?:era)?([0-5])", str(path).lower())
            if match:
                cam_num = int(match.group(1))
                if cam_num in target_cameras:
                    camera_buckets[cam_num].append(str(path))

        total_matched = sum(len(paths) for paths in camera_buckets.values())
        print(f"\n🔍 Discovered {total_matched} total images matching Camera IDs {sorted(list(target_cameras))}:")
        for cam_id in sorted(camera_buckets.keys()):
            print(f"   - Tag '_cam{cam_id}_': {len(camera_buckets[cam_id])} images")

        if total_matched == 0:
            print("⚠️ 0-indexed tags yielded no matches. Retrying search with literal tags {2, 3, 5}...")
            camera_buckets = {cam_id: [] for cam_id in {2, 3, 5}}
            for path in all_image_paths:
                match = re.search(r"_cam([0-5])_", path.name.lower())
                if match and int(match.group(1)) in {2, 3, 5}:
                    camera_buckets[int(match.group(1))].append(str(path))
            total_matched = sum(len(paths) for paths in camera_buckets.values())
            target_cameras = {2, 3, 5}

        if total_matched == 0:
            raise RuntimeError(f"No matching camera images found in {root}")

        rng = np.random.default_rng(seed)
        sampled_paths = []

        if total_matched <= target_size:
            for paths in camera_buckets.values():
                sampled_paths.extend(paths)
        else:
            per_camera_target = target_size // len(target_cameras)
            for cam_id, paths in camera_buckets.items():
                if len(paths) >= per_camera_target:
                    chosen = rng.choice(paths, size=per_camera_target, replace=False).tolist()
                else:
                    chosen = paths
                sampled_paths.extend(chosen)

            remaining_slots = target_size - len(sampled_paths)
            if remaining_slots > 0:
                all_matched = sum(camera_buckets.values(), [])
                unused = list(set(all_matched) - set(sampled_paths))
                extra = rng.choice(unused, size=min(remaining_slots, len(unused)), replace=False).tolist()
                sampled_paths.extend(extra)

        self.paths = sampled_paths
        print(f"Sampled balanced dataset: {len(self.paths)} images for optimized pre-training.\n")

        # NON-GEOMETRIC AUGMENTATION PIPELINE
        # (Preserves exact spatial correspondence across views — required
        #  for the dense per-pixel centroid loss.)
        self.aug = v2.Compose([
            v2.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            v2.ToImage(),
            v2.ToDtype(torch.float32, scale=True),   # -> [0, 1] float tensor
            RandomGrayscaleOrBlur(grayscale_p=0.5),
            AddLightGaussianNoise(std=0.02),
            v2.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        views = [self.aug(img) for _ in range(V)]
        return torch.stack(views, dim=0)


# MULTI-SCALE YOLOv8 BACKBONE (captures P3 / P4 / P5)

class YOLOv8MultiScaleBackbone(nn.Module):
    """
    Wraps the first 10 layers of a YOLOv8 model (the pure backbone,
    pre-FPN) and captures intermediate feature maps at the standard
    P3 (stride 8), P4 (stride 16), and P5 (stride 32) tap points.

    Layer indices match the standard YOLOv8 backbone yaml:
        0: Conv (P1/2)   1: Conv (P2/4)   2: C2f
        3: Conv (P3/8)   4: C2f  <-- P3 tap
        5: Conv (P4/16)  6: C2f  <-- P4 tap
        7: Conv (P5/32)  8: C2f
        9: SPPF          <-- P5 tap
    """
    CAPTURE_INDICES = (4, 6, 9)
    SCALE_NAMES = {4: "P3", 6: "P4", 9: "P5"}

    def __init__(self, weights="yolov8n.pt"):
        super().__init__()
        yolo_model = YOLO(weights).model
        self.layers = nn.ModuleList(list(yolo_model.model[:10]))

    def forward(self, x):
        feats = {}
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i in self.CAPTURE_INDICES:
                feats[self.SCALE_NAMES[i]] = x
        return feats  # {"P3": ..., "P4": ..., "P5": ...}



# DENSE PROJECTOR (per-pixel MLP via 1x1 convolutions)

def build_dense_projector(in_channels, proj_dim):
    return nn.Sequential(
        nn.Conv2d(in_channels, 256, kernel_size=1),
        nn.BatchNorm2d(256),
        nn.ReLU(inplace=True),
        nn.Conv2d(256, 256, kernel_size=1),
        nn.BatchNorm2d(256),
        nn.ReLU(inplace=True),
        nn.Conv2d(256, proj_dim, kernel_size=1),
    )



# DENSE ENCODER — YOLOv8 Multi-Scale Backbone + Per-Scale Projectors

class YOLOv8DenseEncoder(nn.Module):
    def __init__(self, weights="yolov8n.pt", proj_dim=128):
        super().__init__()
        self.backbone = YOLOv8MultiScaleBackbone(weights)

        with torch.no_grad():
            dummy = torch.zeros(1, 3, IMAGE_SIZE, IMAGE_SIZE)
            feats = self.backbone(dummy)

        self.projectors = nn.ModuleDict()
        for name, f in feats.items():
            c_in = f.shape[1]
            self.projectors[name] = build_dense_projector(c_in, proj_dim)
            print(f"{name} | channels: {c_in} -> {proj_dim} | spatial: {tuple(f.shape[-2:])}")

    def forward(self, x):
        # x: (B, V, C, H, W)
        B, Vv = x.shape[:2]
        x = x.flatten(0, 1)  # (B*V, C, H, W)

        feats = self.backbone(x)

        dense_projections = {}
        for name, f in feats.items():
            p = self.projectors[name](f)             # (B*V, proj_dim, Hs, Ws)
            _, D, Hs, Ws = p.shape
            dense_projections[name] = p.view(B, Vv, D, Hs, Ws)

        return dense_projections



# DENSE LeJEPA LOSSES

def dense_lejepa_prediction_loss(proj: torch.Tensor) -> torch.Tensor:
    """
    proj: (B, V, D, H, W)
    Local centroid is computed per pixel location (mean over the view
    axis, dim=1), independently for every (h, w). Because augmentations
    are non-geometric, pixel (h, w) means the same physical location in
    every view, so this centroid is a meaningful per-location target.
    """
    mu = proj.mean(dim=1, keepdim=True)
    dif = mu - proj
    return dif.square().mean()


def subsample_spatial(proj: torch.Tensor, max_locations: int) -> torch.Tensor:
    """proj: (B, V, D, H, W) -> (B, V, D, N) with N <= max_locations."""
    B, Vv, D, H, W = proj.shape
    flat = proj.reshape(B, Vv, D, H * W)
    if H * W > max_locations:
        idx = torch.randperm(H * W, device=proj.device)[:max_locations]
        flat = flat[..., idx]
    return flat


def prepare_for_sigreg(proj_flat: torch.Tensor) -> torch.Tensor:
    """(B, V, D, N) -> (V, B*N, D) for the SIGReg statistic."""
    B, Vv, D, N = proj_flat.shape
    return proj_flat.permute(1, 0, 3, 2).contiguous().reshape(Vv, B * N, D)


def compute_multiscale_dense_losses(dense_projections, sigreg, max_sigreg_locations=MAX_SIGREG_LOCATIONS):
    pred_by_scale, sig_by_scale = {}, {}
    for name, proj in dense_projections.items():
        pred_by_scale[name] = dense_lejepa_prediction_loss(proj)
        sig_input = prepare_for_sigreg(subsample_spatial(proj, max_sigreg_locations))
        sig_by_scale[name] = sigreg(sig_input)

    pred_loss = torch.stack(list(pred_by_scale.values())).mean()
    sig_loss = torch.stack(list(sig_by_scale.values())).mean()
    return pred_loss, sig_loss, pred_by_scale, sig_by_scale



# 3D PCA EMBEDDING LOGGING (spatially-pooled, multi-scale concat)

@torch.no_grad()
def save_interactive_pca(net, loader, epoch, out_dir: Path, max_batches=60):
    net.eval()
    feats = []
    batches_seen = 0

    for vs in loader:
        vs = vs.to(DEVICE)
        dense = net(vs)

        pooled_scales = []
        for name, proj in dense.items():
            pooled = proj.mean(dim=(-1, -2))  # (B, V, D) spatial average
            pooled_scales.append(pooled)

        combined = torch.cat(pooled_scales, dim=-1).reshape(-1, sum(p.shape[-1] for p in pooled_scales))
        feats.append(combined.cpu().numpy())
        batches_seen += 1
        if batches_seen >= max_batches:
            break

    X = np.concatenate(feats, axis=0)

    print(f"\n--- [EMBEDDINGS INSPECTION - EPOCH {epoch + 1}] ---")
    print(f"Embeddings Matrix Shape: {X.shape}")
    print(f"Embeddings Mean: {X.mean():.4f} | Std Dev: {X.std():.4f}")
    print(f"Sample Features:\n{X[0, :5]}\n" + "-" * 50)

    Z = PCA(n_components=3, random_state=0).fit_transform(X)

    fig = px.scatter_3d(
        x=Z[:, 0], y=Z[:, 1], z=Z[:, 2],
        opacity=0.7,
        title=f"YOLOv8 Dense LeJEPA (Cam 2,3,5) — Epoch {epoch + 1}",
    )
    fig.update_traces(marker=dict(size=3))
    fig.update_layout(margin=dict(l=0, r=0, t=40, b=0))

    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"pca_epoch_{epoch+1:03d}.html"
    fig.write_html(str(out_path), include_plotlyjs="cdn")
    print(f" Saved 3D PCA visualization: {out_path}")


def save_loss_history(history, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(history)
    df.insert(0, "epoch", np.arange(1, len(df) + 1))
    out_path = out_dir / "loss_history_50k_cam235_dense.csv"
    df.to_csv(out_path, index=False)
    print(f" Saved loss history CSV: {out_path}")



# BACKBONE EXPORT — reload pretrained weights into a clean YOLO

def load_pretrained_backbone_into_yolo(checkpoint_path, yolo_weights="yolov8n.pt", proj_dim=PROJ_DIM):
    """
    Loads a saved YOLOv8DenseEncoder checkpoint and transplants only the
    backbone weights (layers 0-9) into a fresh ultralytics YOLO model.
    The dense projection heads are pretext-only and intentionally discarded
    — your detection/segmentation head will be freshly initialized on top
    of these pretrained backbone weights.
    """
    encoder = YOLOv8DenseEncoder(weights=yolo_weights, proj_dim=proj_dim)
    state = torch.load(checkpoint_path, map_location="cpu")
    encoder.load_state_dict(state)

    yolo = YOLO(yolo_weights)
    for i in range(10):
        yolo.model.model[i].load_state_dict(encoder.backbone.layers[i].state_dict())

    print(f"Loaded pretrained backbone (layers 0-9) from {checkpoint_path} into fresh YOLO model.")
    return yolo



# MAIN EXECUTION LOOP

def main():
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    ckpt_dir = Path(CHECKPOINT_DIR)
    plots_dir = ckpt_dir / "interactive_plots"
    pca_dir = plots_dir / "pca_3d"

    torch.manual_seed(42)

    dataset = Cotton50kSamplerDataset(
        root=JPEG_ROOT,
        target_cameras=TARGET_CAMERAS_0INDEX,
        target_size=TARGET_DATASET_SIZE,
        seed=42
    )

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

    net = YOLOv8DenseEncoder(weights="yolov8n.pt", proj_dim=PROJ_DIM).to(DEVICE)
    sigreg = SIGReg().to(DEVICE)

    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    scale_names = list(YOLOv8MultiScaleBackbone.SCALE_NAMES.values())
    history = {"lejepa_total": [], "pred_avg": [], "sigreg_avg": []}
    for name in scale_names:
        history[f"pred_{name}"] = []
        history[f"sigreg_{name}"] = []

    for epoch in range(EPOCHS):
        net.train()
        ep_total, ep_pred, ep_sig, n_batches = 0.0, 0.0, 0.0, 0
        ep_pred_scale = {name: 0.0 for name in scale_names}
        ep_sig_scale = {name: 0.0 for name in scale_names}

        pbar = tqdm.tqdm(loader, desc=f"Pre-training Epoch {epoch+1}/{EPOCHS}")
        for vs in pbar:
            vs = vs.to(DEVICE)

            dense_proj = net(vs)
            pred_loss, sig_loss, pred_by_scale, sig_by_scale = compute_multiscale_dense_losses(dense_proj, sigreg)

            loss = (1.0 - LAMBDA) * pred_loss + LAMBDA * sig_loss

            opt.zero_grad()
            loss.backward()
            opt.step()

            ep_total += float(loss.detach().cpu())
            ep_pred += float(pred_loss.detach().cpu())
            ep_sig += float(sig_loss.detach().cpu())
            for name in scale_names:
                ep_pred_scale[name] += float(pred_by_scale[name].detach().cpu())
                ep_sig_scale[name] += float(sig_by_scale[name].detach().cpu())
            n_batches += 1

        ep_total /= max(1, n_batches)
        ep_pred /= max(1, n_batches)
        ep_sig /= max(1, n_batches)
        for name in scale_names:
            ep_pred_scale[name] /= max(1, n_batches)
            ep_sig_scale[name] /= max(1, n_batches)

        history["lejepa_total"].append(ep_total)
        history["pred_avg"].append(ep_pred)
        history["sigreg_avg"].append(ep_sig)
        for name in scale_names:
            history[f"pred_{name}"].append(ep_pred_scale[name])
            history[f"sigreg_{name}"].append(ep_sig_scale[name])

        scale_summary = " | ".join(f"{n}: pred={ep_pred_scale[n]:.5f} sig={ep_sig_scale[n]:.5f}" for n in scale_names)
        print(
            f"Epoch {epoch+1:03d} | Total: {ep_total:.6f} | Pred(avg): {ep_pred:.6f} | "
            f"SIGReg(avg): {ep_sig:.6f} | {scale_summary}"
        )

        if epoch in PCA_EPOCHS:
            save_interactive_pca(net, loader, epoch, pca_dir)

    ckpt_path = ckpt_dir / "dense_lejepa_yolov8_cotton_50k_checkpoint.pth"
    torch.save(net.state_dict(), ckpt_path)
    print(f"Optimized pre-trained checkpoint saved to: {ckpt_path}")

    save_loss_history(history, plots_dir)


if __name__ == "__main__":
    main()


 Device: Apple Silicon GPU via MPS (mps)

🔍 Discovered 200276 total images matching Camera IDs [1, 2, 4]:
   - Tag '_cam1_': 85717 images
   - Tag '_cam2_': 85717 images
   - Tag '_cam4_': 28842 images
Sampled balanced dataset: 50000 images for optimized pre-training.

P3 | channels: 64 -> 128 | spatial: (16, 16)
P4 | channels: 128 -> 128 | spatial: (8, 8)
P5 | channels: 256 -> 128 | spatial: (4, 4)


Pre-training Epoch 1/60: 100%|██████████| 3125/3125 [14:47<00:00,  3.52it/s]


Epoch 001 | Total: 1.277622 | Pred(avg): 0.209357 | SIGReg(avg): 5.550678 | P3: pred=0.17463 sig=7.66250 | P4: pred=0.19493 sig=5.92856 | P5: pred=0.25851 sig=3.06098

--- [EMBEDDINGS INSPECTION - EPOCH 1] ---
Embeddings Matrix Shape: (1920, 384)
Embeddings Mean: -0.0015 | Std Dev: 0.3060
Sample Features:
[    0.12831    0.022425    -0.14652     0.20878    -0.25187]
--------------------------------------------------
 Saved 3D PCA visualization: checkpoints_cam235_50k_dense_lejepa/interactive_plots/pca_3d/pca_epoch_001.html


Pre-training Epoch 2/60: 100%|██████████| 3125/3125 [15:15<00:00,  3.41it/s]


Epoch 002 | Total: 0.705870 | Pred(avg): 0.215489 | SIGReg(avg): 2.667394 | P3: pred=0.18989 sig=3.72572 | P4: pred=0.20829 sig=2.46295 | P5: pred=0.24829 sig=1.81351


...

Epoch 058 | Total: 0.355439 | Pred(avg): 0.167754 | SIGReg(avg): 1.106178 | P3: pred=0.15065 sig=1.33994 | P4: pred=0.15127 sig=1.00334 | P5: pred=0.20134 sig=0.97525


Pre-training Epoch 59/60: 100%|██████████| 3125/3125 [11:05<00:00,  4.70it/s]


Epoch 059 | Total: 0.355484 | Pred(avg): 0.168026 | SIGReg(avg): 1.105315 | P3: pred=0.15084 sig=1.33773 | P4: pred=0.15140 sig=1.00241 | P5: pred=0.20184 sig=0.97581


Pre-training Epoch 60/60: 100%|██████████| 3125/3125 [10:56<00:00,  4.76it/s]


Epoch 060 | Total: 0.354178 | Pred(avg): 0.166930 | SIGReg(avg): 1.103171 | P3: pred=0.15002 sig=1.33415 | P4: pred=0.15042 sig=1.00217 | P5: pred=0.20035 sig=0.97319

--- [EMBEDDINGS INSPECTION - EPOCH 60] ---
Embeddings Matrix Shape: (1920, 384)
Embeddings Mean: 0.0024 | Std Dev: 0.2093
Sample Features:
[  -0.086331   -0.046034    0.013297     0.16076   -0.097757]
--------------------------------------------------
 Saved 3D PCA visualization: checkpoints_cam235_50k_dense_lejepa/interactive_plots/pca_3d/pca_epoch_060.html
Optimized pre-trained checkpoint saved to: checkpoints_cam235_50k_dense_lejepa/dense_lejepa_yolov8_cotton_50k_checkpoint.pth
 Saved loss history CSV: checkpoints_cam235_50k_dense_lejepa/interactive_plots/loss_history_50k_cam235_dense.csv
